# Scaled Dot-Product Attention

[← Back to wiki](https://ml-viz.vercel.app/wiki/scaled-dot-product-attention)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('dark_background')

## From-scratch attention

In [ ]:
def softmax(x, axis=-1):
    e = np.exp(x - x.max(axis=axis, keepdims=True))  # numerically stable
    return e / e.sum(axis=axis, keepdims=True)

def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)     # (n, n)
    if mask is not None:
        scores[mask] = -1e9
    weights = softmax(scores)            # row-wise softmax
    return weights @ V, weights

# Wiki worked example: 3 tokens, d_k=2
Q = np.array([[2,0],[0,2],[2,2]], dtype=float)
K = np.array([[2,0],[0,2],[1,1]], dtype=float)
V = np.array([[1,0],[0,1],[10,10]], dtype=float)

out, W = scaled_dot_product_attention(Q, K, V)
print("Attention weights (3×3):")
print(W.round(3))
print("\nOutput (3×2):")
print(out.round(3))
# Verify: row 1 ≈ [2.635, 1.912]; row 3 ≈ [3.667, 3.667]

## Variance grows with d_k — the √d_k fix

In [ ]:
d_values = [2, 8, 32, 64, 128]
n_samples = 20_000
variances = []
for d in d_values:
    q = np.random.randn(n_samples, d)
    k = np.random.randn(n_samples, d)
    dots = (q * k).sum(axis=1)
    variances.append(dots.var())

plt.figure(figsize=(7,4))
plt.plot(d_values, variances, 'o-', color='#6366f1', label='Empirical Var(q·k)')
plt.plot(d_values, d_values, '--', color='#f59e0b', label='Expected = d_k')
plt.xlabel('d_k'); plt.ylabel('Variance'); plt.title('Why we scale by 1/√d_k')
plt.legend(); plt.tight_layout(); plt.show()

## Attention heatmap on a 4-token sequence

In [ ]:
rng = np.random.default_rng(42)
n, d_k = 6, 8
Q_r = rng.standard_normal((n, d_k))
K_r = rng.standard_normal((n, d_k))
V_r = rng.standard_normal((n, d_k))

_, W_r = scaled_dot_product_attention(Q_r, K_r, V_r)

tokens = ['The','cat','sat','on','the','mat']
plt.figure(figsize=(6,5))
plt.imshow(W_r, vmin=0, vmax=1, cmap='Blues', aspect='auto')
plt.colorbar(label='Attention weight')
plt.xticks(range(n), tokens, rotation=45)
plt.yticks(range(n), tokens)
plt.title('Attention weights (random projections)')
plt.tight_layout(); plt.show()

## ✏️ Your turn

**Task:** Implement **causal masking** — add $-\infty$ to all upper-triangle entries of the score matrix before softmax, so token $t$ can only attend to positions $\le t$.

Verify that after masking, the upper triangle of the weight matrix is exactly 0.

In [ ]:
# TODO(you): create a causal mask (upper triangle, diagonal=1)
# mask = np.triu(np.ones((n, n), dtype=bool), k=1)
# out_causal, W_causal = scaled_dot_product_attention(Q_r, K_r, V_r, mask=mask)

In [ ]:
# assert np.allclose(W_causal[np.triu(np.ones((n,n),dtype=bool), k=1)], 0, atol=1e-6)

<details><summary>Solution</summary>

```python
mask = np.triu(np.ones((n,n), dtype=bool), k=1)
out_c, W_c = scaled_dot_product_attention(Q_r, K_r, V_r, mask=mask)
print('Upper-triangle weights (should be ~0):')
print(W_c[mask].round(6))
assert np.allclose(W_c[mask], 0, atol=1e-6)
print('Causal mask verified ✓')
```
</details>